In [19]:
from datasets import load_dataset
from datasets import load_from_disk

import torch
from torch import Tensor
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

import os
from typing import List
import sentencepiece as spm
import yaml

from transformer.model import Transformer
from transformer.scheduler import TransformerLRScheduler

In [20]:
def load_data(file_path: str)->List[str]:
    data = []
    with open(file_path, "r", encoding="utf-8") as f:

        for sentence in f.readlines():
            # 去除句首句尾空白字符
            sentence = sentence.strip()
            # 去除尖括号开头的标签
            if sentence[0] == "<":
                continue
            data.append(sentence)
    return data
DATA_PATH = "D:/Learn/machine_learning/data/iwslt2017-en-de/en-de"
train_en = load_data(DATA_PATH + "/train.tags.en-de.en")
train_de = load_data(DATA_PATH + "/train.tags.en-de.de")
# print(train_en[:1], train_de[:1], sep="\n")
with open("configs.yaml", "r", encoding="utf-8") as f:
    configs = yaml.safe_load(f)

In [ ]:
if not configs["BPE"]["IS_TRAINED"]:
    # 写入合并语料（每行一句，混合源和目标）
    with open("bpe_corpus.txt", "w", encoding="utf-8") as f:
        for en_sent, de_sent in zip(train_en, train_de):
            f.write(en_sent + "\n")
            f.write(de_sent + "\n")
    # 训练 BPE 模型
    spm.SentencePieceTrainer.train(
        input="bpe_corpus.txt",
        model_prefix="bpe_shared",         # 输出: bpe_shared.model 和 bpe_shared.vocab
        vocab_size=configs["BPE"]["VOCAB_SIZE"],                  # BPE 词汇量（IWSLT 常用 32k）
        character_coverage=configs["BPE"]["CHARACTER_COVERAGE"],  # 覆盖德语的特殊字符（ä ü ö ß）
        model_type=configs["BPE"]["MODEL_TYPE"],                  # BPE 算法
        num_threads=configs["BPE"]["NUM_THREADS"],
        pad_id=configs["BPE"]["PAD_ID"],                          # <pad> 索引 0
        unk_id=configs["BPE"]["UNK_ID"],                          # <unk> 索引 1
        bos_id=configs["BPE"]["BOS_ID"],                          # <s>  索引 2
        eos_id=configs["BPE"]["EOS_ID"],                          # </s> 索引 3
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<s>",
        eos_piece="</s>",
    )
    configs["BPE"]["IS_TRAINED"] = True
    with open("configs.yaml", "w", encoding="utf-8") as f:
        yaml.dump(configs, f, allow_unicode=True, sort_keys=False)

In [21]:
# 加载模型
sp = spm.SentencePieceProcessor()
sp.load("bpe_shared.model")
# 测试
# print(sp.encode("Thank you so much.", out_type=str))
# → ['▁Thank', '▁you', '▁so', '▁much', '.']

True

In [22]:
# BPE分词
emb_en = []
emb_de = []
for en, de in zip(train_en, train_de):
    emb_en.append(sp.encode(en, out_type=int))
    emb_de.append(sp.encode(de, out_type=int))

In [23]:
# 短句按语序语境分组合并
combine_en = []
combine_de = []
merge_sz = configs["BPE"]["MERGE_SIZE"]
for start in range(0, len(train_en), merge_sz):
    end = min(start + merge_sz, len(train_en))
    combine_en.append([token for group in emb_en[start: end] for token in group])
    combine_de.append([token for group in emb_de[start: end] for token in group])

In [24]:
# ============================================================
# 超参数配置
# ============================================================
os.environ["CUDA_LAUNCH_BLOCKING"]="1"
LAYER_NUM = configs['MODEL']['LAYER_NUM']
D_MODEL = configs['MODEL']['D_MODEL']
HEAD = configs['MODEL']['HEAD']
EPS = configs['MODEL']['EPS']
D_FF = configs['MODEL']['D_FF']
DROPOUT = configs['MODEL']['DROPOUT']
MAX_LENGTH = configs['MODEL']['MAX_LENGTH']
BETA1 = configs['LR']['BETA1']
BETA2 = configs['LR']['BETA2']
L_EPS = configs['LR']['L_EPS']
WARMUP = configs['LR']['WARMUP']
EPOCHS = configs['TRAIN']['EPOCHS']
BATCH_SIZE = configs['TRAIN']['BATCH_SIZE']
VOCAB_SZ = configs['BPE']['VOCAB_SIZE']
PAD_ID = configs['BPE']['PAD_ID']
EOS_ID = configs['BPE']['EOS_ID']
# ============================================================
# 设备
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Current device: ",device)
# ============================================================
# 模型、优化器、调度器、损失函数
# ============================================================
tf = Transformer(
    layer_num=LAYER_NUM,
    d_model=D_MODEL,
    src_emb_sz=VOCAB_SZ,
    tgt_emb_sz=VOCAB_SZ,
    head=HEAD,
    eps=EPS,
    d_ff=D_FF,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.Adam(
    params=tf.parameters(), betas=(BETA1, BETA2), eps=L_EPS)
lr_scheduler = TransformerLRScheduler(
    optimizer=optimizer, d_model=D_MODEL, warmup_steps=WARMUP)
# 平滑分布只覆盖非 PAD、非 EOS 的 token
smooth_indices = [i for i in range(VOCAB_SZ)
                  if i != PAD_ID and i != EOS_ID]
smooth_indices = torch.tensor(smooth_indices, device=device)

Current device:  cuda


In [25]:
# 创建Dataset和DataLoader
class TranslationDataset(Dataset):
    def __init__(self, src_data, tgt_data):
        self.src_data = src_data
        self.tgt_data = tgt_data

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.src_data[idx], dtype=torch.long),
            torch.tensor(self.tgt_data[idx], dtype=torch.long)
        )

def collate_fn(batch):
    src_batch = []
    tgt_batch = []
    for src, tgt in batch:
        src_batch.append(src)
        tgt_batch.append(tgt)
    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=PAD_ID
    )
    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=PAD_ID
    )

    src_pad_mask = (src_batch == PAD_ID)
    tgt_pad_mask = (tgt_batch == PAD_ID)

    return src_batch, tgt_batch, src_pad_mask, tgt_pad_mask

# 损失函数的label_smooth
def smoothed_loss(logits: Tensor, targets: Tensor,
                  ignore_index: int, smooth_indices: Tensor, epsilon: float = 0.1):
    """
    logits: [N, V]  模型输出
    targets: [N]    整数 token ID
    ignore_index:   pad 位置不计算 loss
    smooth_indices: 参与平滑均匀分布的 token ID 列表（不含 PAD、EOS）
    epsilon:        平滑强度
    """
    mask = (targets != ignore_index)
    logits = logits[mask]
    targets = targets[mask]
    # 正确标签的 one-hot
    true_dist = torch.zeros_like(logits)
    true_dist.scatter_(1, targets.unsqueeze(1), 1.0)
    # 平滑背景：只在 smooth_indices 上均匀分布
    smooth_dist = torch.zeros_like(logits)
    smooth_dist[:, smooth_indices] = 1.0 / len(smooth_indices)
    # 混合
    target_dist = true_dist * (1 - epsilon) + smooth_dist * epsilon
    log_probs = F.log_softmax(logits, dim=-1)
    return -(target_dist * log_probs).sum(dim=-1).mean()

dataset = TranslationDataset(combine_en, combine_de)
loader = DataLoader(
    dataset,
    batch_size=configs["TRAIN"]["BATCH_SIZE"],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

In [ ]:
# ============================================================
# 训练循环
# ============================================================

start_epoch = 0
if os.path.exists('transformer.pt'):
    print("Resuming from checkpoint...")
    checkpoint = torch.load('transformer.pt', map_location=device)
    tf.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    lr_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint.get('current_epoch', 0)
    print(f"Resumed at epoch {start_epoch}, step_num={lr_scheduler.step_num}")

for epoch in range(start_epoch, start_epoch + EPOCHS):
    epoch_loss = 0
    cnt = 0
    for batch in loader:
        src, tgt, src_pad_mask, tgt_pad_mask = batch
        src = src.to(device)
        tgt = tgt.to(device)
        src_pad_mask = src_pad_mask.to(device)
        tgt_pad_mask = tgt_pad_mask.to(device)

        optimizer.zero_grad()

        y = tf(src, tgt[:, :-1], src_pad_mask, tgt_pad_mask[:, :-1])

        # loss = criterion(y.reshape(-1, VOCAB_SZ), tgt[:, 1:].reshape(-1))
        loss = smoothed_loss(
            y.reshape(-1, VOCAB_SZ),
            tgt[:, 1:].reshape(-1),
            ignore_index=PAD_ID,
            smooth_indices=smooth_indices,
            epsilon=0.1
        )

        if torch.isnan(loss):
            for x in batch:
                print(sp.decode(src))
                print(sp.decode(tgt))
                print(y.reshape(-1, VOCAB_SZ))
                print(tgt[:, 1:].reshape(-1))
                input()
        if (cnt + 1) % 50 == 0:
            print(f"Batch No: {cnt + 1}, loss: {loss.item()}")

        epoch_loss += loss.item()
        cnt += 1

        loss.backward()

        # 梯度裁剪：将梯度的全局 L2 范数限制在 max_norm 以内
        # torch.nn.utils.clip_grad_norm_(tf.parameters(), max_norm=1.0)

        # 基于历史百分位的动态阈值（AutoClip）
        # Seethrough 等人提出的方法，每 N 步统计历史梯度范数的 p90 分位数作为阈值：
        # auto_clipper.clip(tf.parameters())

        # AGC（Adaptive Gradient Clipping，自适应梯度裁剪）
        # 不要用固定的梯度阈值裁剪，而是根据参数自身的大小动态决定梯度允许的最大范围。
        # agc_grad_clipper.clip(tf.parameters())

        optimizer.step()
        lr_scheduler.step()
    print(f"Epoch: {epoch}; Average loss: {epoch_loss / cnt}")

# ============================================================
# 保存模型
# ============================================================
checkpoint = {
    'model_state_dict': tf.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': lr_scheduler.state_dict(),
    'current_epoch': start_epoch + EPOCHS,
}
torch.save(checkpoint, "transformer.pt")
print("Model saved to transformer.pt")
# 已连续运行的epoch: